# Build Different Constellations with satComTopology
This tutorial is for people who want to create a simulation **without touching the library code**, only through configuration. You will learn:
- the shape of a `SimulationProperty` configuration (the same object whether you write it as a Python `dict` or as a YAML file)
- how to build a **Walker Delta** constellation (global coverage, e.g. Starlink / OneWeb style)
- how to build a **Walker Star** constellation (near-polar coverage, e.g. Iridium style)
- how to attach Ground Stations from a file
- how to load the exact same configuration from a YAML file instead of a Python `dict`
- how to export a simulation snapshot to NetworkX and to Cesium

For the fully manual, low level way of building a simulation object by object, see [deeply_understand_sat_com_topology.ipynb](deeply_understand_sat_com_topology.ipynb). For a hands-on lab computing shortest paths, see [shortest_path_lab.ipynb](shortest_path_lab.ipynb).

## Setup
Make sure your virtual environment is active (see the project [README](../README.md)) and that `sat_com_topology` is installed:
```shell
pip install git+https://github.com/elbarbi/satComTopology.git@develop#egg=sat_com_topology
```
This tutorial uses version `4.2.x` of the library. You can check your installed version with `pip show sat_com_topology`.

## The configuration model
Every simulation is described by a single `SimulationProperty` object (`sat_com_builder.models.SimulationProperty`). It has two main sections:

- **`walker_shells`**: a list of satellite shells. Each shell has:
  - `type`: `"delta"` (satellites spread over 360°, best for global coverage) or `"star"` (satellites spread over 180°, best for near-polar coverage)
  - `constellation_property`: orbit planes, satellites per plane, inclination, mean revolutions per day, ...
  - `orbital_connectivity_property`: how satellites connect to each other (inter-satellite links) and how many ground objects they can serve
- **`ground_objects_properties`**: a list of ground object groups (ground stations, user terminals, or points of presence), each pointing to a data file and a `connectivity_properties` block describing how they connect to satellites (minimum elevation angle, connection strategy, ...)

We will build two shells with the exact same ground stations, only changing the `type`, `inclination` and orbit count to see the difference.

In [ ]:
from sat_com_adapter.adapters import NetworkXAdapter, CesiumAdapter
from sat_com_builder.configuration_manager import BaseConfigurationManager
from sat_com_builder.models import SimulationProperty


## 1. A Walker Delta constellation
A Walker Delta constellation spreads its orbital planes over 360°. It is the shape used by most global broadband constellations (Starlink, OneWeb, ...).

We attach a single Ground Station group loaded from [`ground_stations.txt`](configurations/ground_stations.txt), using the `everything-visible` strategy (connect to every satellite currently above the horizon).

In [ ]:
walker_delta_config = {
    "simulation_name": "Walker Delta Example",
    "start_date": "2026-01-01 00:00:00.000000",
    "end_date": "2026-01-01 00:00:10.000000",
    "movement_model": "pyorbital",
    "distance_model": "sklearn",
    "ground_objects_properties": [
        {
            "identifier": "Example Ground Stations",
            "data_file": "./configurations/ground_stations.txt",
            "type": "ground_station",
            "connectivity_properties": {
                "elevation_above_horizon": 20,
                "ground_to_space_connections_strategy": "everything-visible",
            },
        },
    ],
    "walker_shells": [
        {
            "type": "delta",
            "constellation_property": {
                "identifier": "Delta Walker",
                "amount_of_orbit_plane": 12,
                "amount_of_satellite_per_orbit_plane": 22,
                "inclination": 70.0,
                "phase_difference_between_satellites": True,
                "mean_revolution_per_day": 15.0,
            },
            "orbital_connectivity_property": {
                "adjacent_inter_satellite_shifting": 0,
                "maximum_inter_satellite_count": 4,
                "maximum_inter_satellite_range_distance": 1000,
                "maximum_ground_station_range": 5000,
                "maximum_user_terminal_range": 1000,
                "maximum_connected_ground_object": 10000,
                "maximum_connected_user_terminal": 1000,
                "maximum_connected_ground_station": 10,
            },
            "ground_object_white_list": [],
        }
    ],
}

delta_simulation_properties = SimulationProperty(**walker_delta_config)
delta_configuration_manager = BaseConfigurationManager(simulation_property=delta_simulation_properties)

delta_simulation_manager = delta_configuration_manager.load_simulation()

print(f"{len(delta_simulation_manager.get_satellites())} satellites")
print(f"{len(delta_simulation_manager.get_ground_stations())} ground stations")
print(f"{len(delta_simulation_manager.get_inter_satellites_links())} inter-satellite links")
print(f"{len(delta_simulation_manager.get_ground_stations_links())} ground station links")


## 2. A Walker Star constellation
A Walker Star constellation spreads its orbital planes over only 180°, which packs planes closer together near the poles. It is typically used for near-polar constellations (Iridium style).

We reuse the exact same ground stations and connectivity rules, and only change `type` to `"star"`, together with a near-polar inclination.

In [ ]:
walker_star_config = dict(walker_delta_config)
walker_star_config["simulation_name"] = "Walker Star Example"
walker_star_config["walker_shells"] = [
    {
        "type": "star",
        "constellation_property": {
            "identifier": "Star Walker",
            "amount_of_orbit_plane": 6,
            "amount_of_satellite_per_orbit_plane": 11,
            "inclination": 86.4,
            "phase_difference_between_satellites": True,
            "mean_revolution_per_day": 14.34,
        },
        "orbital_connectivity_property": {
            "adjacent_inter_satellite_shifting": 0,
            "maximum_inter_satellite_count": 4,
            "maximum_inter_satellite_range_distance": 1000,
            "maximum_ground_station_range": 5000,
            "maximum_user_terminal_range": 1000,
            "maximum_connected_ground_object": 10000,
            "maximum_connected_user_terminal": 1000,
            "maximum_connected_ground_station": 10,
        },
        "ground_object_white_list": [],
    }
]

star_simulation_properties = SimulationProperty(**walker_star_config)
star_configuration_manager = BaseConfigurationManager(simulation_property=star_simulation_properties)

star_simulation_manager = star_configuration_manager.load_simulation()

print(f"{len(star_simulation_manager.get_satellites())} satellites")
print(f"{len(star_simulation_manager.get_ground_stations())} ground stations")
print(f"{len(star_simulation_manager.get_inter_satellites_links())} inter-satellite links")
print(f"{len(star_simulation_manager.get_ground_stations_links())} ground station links")


Both simulations are built from the same `walker_delta_config` dict, only the `walker_shells` section changed. This is the pattern to follow whenever you want to experiment with a new constellation shape: keep the ground segment fixed and vary the space segment (or the other way around).

## 3. Loading a configuration from a YAML file
Writing your configuration as a plain Python `dict` is convenient for experimentation, but you can also store it in a YAML file. This is useful to keep a versioned, shareable configuration.

Have a look at [`configurations/configuration_example.yaml`](configurations/configuration_example.yaml), it describes the exact same kind of Walker Delta shell as above.

> **Note:** `YamlConfigurationManager` in `sat_com_topology==4.2.0` forwards the raw YAML `dict` to `BaseConfigurationManager` without validating it against `SimulationProperty`, which currently raises an `AttributeError`. Until this is fixed upstream, load the YAML yourself and validate it explicitly, as shown below — it is also generally the safer pattern since you get Pydantic's validation errors immediately.

In [ ]:
import yaml

with open("configurations/configuration_example.yaml") as config_file:
    yaml_config = yaml.safe_load(config_file)

yaml_simulation_properties = SimulationProperty(**yaml_config)
yaml_configuration_manager = BaseConfigurationManager(simulation_property=yaml_simulation_properties)

yaml_simulation_manager = yaml_configuration_manager.load_simulation()

print(f"{len(yaml_simulation_manager.get_satellites())} satellites loaded from YAML")


## 4. Export your simulation
### Export to NetworkX
You can export the topology as a NetworkX graph, either to use it directly in python or to save it as JSON.

In [ ]:
from pathlib import Path

Path("results").mkdir(exist_ok=True)

networkx_adapter = NetworkXAdapter(delta_simulation_manager)

graph = networkx_adapter.create_full_networkx_graph(export_object_position=True, export_link_length=True)
print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

networkx_adapter.adapt(output_directory="results/walker_delta_example-networkx.json")


### Visualize with Cesium
Cesium renders your constellation as an interactive 3D web page. You need a free [Cesium Ion token](https://ion.cesium.com/tokens?page=1) to display the Earth imagery, but the HTML page will still be generated without one (it will just render an empty globe).

In [ ]:
CESIUM_TOKEN = "<YOUR_CESIUM_TOKEN>"

cesium_adapter = CesiumAdapter(delta_simulation_manager)
cesium_adapter.build_renderer_simulation_with_links(CESIUM_TOKEN)
cesium_adapter.adapt(output_directory="results/walker_delta_example-cesium.html")


Open [`results/walker_delta_example-cesium.html`](results/walker_delta_example-cesium.html) in your browser to see the result.

## Next steps
- To build a simulation object by object without any configuration file, go to [deeply_understand_sat_com_topology.ipynb](deeply_understand_sat_com_topology.ipynb).
- To compute a shortest path between a User Terminal and a Ground Station, go to [shortest_path_lab.ipynb](shortest_path_lab.ipynb).